In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip install -q -U transformers accelerate opencv-python-headless

In [3]:
import sys
from pathlib import Path
D = Path("/kaggle/input/datasets/arnobsaha1/weightgait-pilot")
sys.path.append(str(D))
print([(D / f).exists() for f in ("clip_meta.json", "prompts_v2.py", "gait_parse.py")])  # must be [True, True, True]
print(len(list((D / "kaggle_representations").iterdir())), "clip folders")                # expect 60

import run_gemma
from samples import DEV_INSTANCES
print(len(DEV_INSTANCES), "dev clips")                                                    # expect 30

[True, True, True]
89 clip folders
30 dev clips


In [5]:
import gemma_backend
gemma_backend.MAX_SOFT_TOKENS = 140
print(gemma_backend.MAX_SOFT_TOKENS)   # must print 140

140


In [6]:
import shutil
from pathlib import Path
out = Path("/kaggle/working/results/gemma_runs.jsonl")
if out.exists():
    shutil.move(out, "/kaggle/working/results/gemma_runs_280_discard.jsonl")

In [7]:
run_gemma.run(DEV_INSTANCES, ["raw_video", "skeleton", "overlay", "montage", "blind"],
              ["R_rubric", "Z_zero_shot"], [1])

0 calls on disk, 242 to run


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[1/242] R_rubric 1fps raw_video 427: gt=0 pred=0 (OK, 3 img, 48s)
[2/242] R_rubric 1fps raw_video 548: gt=0 pred=0 (OK, 5 img, 41s)
[14/242] R_rubric 1fps raw_video 745: gt=1 pred=0 (WRONG, 4 img, 40s)
[15/242] R_rubric 1fps raw_video 920: gt=1 pred=0 (WRONG, 5 img, 41s)
[16/242] R_rubric 1fps raw_video 1220: gt=1 pred=0 (WRONG, 4 img, 39s)
[17/242] R_rubric 1fps raw_video 1344: gt=1 pred=0 (WRONG, 4 img, 40s)
[18/242] R_rubric 1fps raw_video 1584: gt=1 pred=0 (WRONG, 5 img, 42s)
[19/242] R_rubric 1fps raw_video 1642: gt=1 pred=0 (WRONG, 4 img, 39s)
[20/242] R_rubric 1fps raw_video 1775: gt=1 pred=0 (WRONG, 5 img, 41s)
[21/242] R_rubric 1fps raw_video 467: gt=2 pred=0 (WRONG, 4 img, 40s)
[22/242] R_rubric 1fps raw_video 582: gt=2 pred=2 (OK, 7 img, 45s)
[23/242] R_rubric 1fps raw_video 703: gt=2 pred=0 (WRONG, 5 img, 41s)
[24/242] R_rubric 1fps raw_video 770: gt=2 pred=2 (OK, 4 img, 41s)
[25/242] R_rubric 1fps raw_video 957: gt=2 pred=2 (OK, 8 img, 47s)
[26/242] R_rubric 1fps raw_video

In [ ]:
from samples import DEV_INSTANCES, HELD_OUT_INSTANCES
import cv2, numpy as np
keep = {str(i) for i in DEV_INSTANCES + HELD_OUT_INSTANCES}
folders = {p.name for p in (D / "kaggle_representations").iterdir()}
print("extra folders:", len(folders - keep), "| missing:", sorted(keep - folders))

def colour_pixels(path):
    cap = cv2.VideoCapture(str(path)); ok, f = cap.read(); cap.release()
    b, g, r = f[..., 0].astype(int), f[..., 1].astype(int), f[..., 2].astype(int)
    blue = ((b > 200) & (r < 100) & (g < 140)).sum()   # BGR (255,80,0) = left, blue
    red = ((r > 200) & (b < 100) & (g < 100)).sum()    # BGR (0,0,255) = right, red
    return int(blue), int(red)

for clip in (442, 427, 467):
    for rep in ("skeleton", "overlay"):
        print(clip, rep, "blue/red pixels:", colour_pixels(D / "kaggle_representations" / str(clip) / f"{rep}.mp4"))

In [ ]:
def colour_pixels(path, n=12):
    cap = cv2.VideoCapture(str(path)); tot = int(cap.get(7)); bl = rd = wh = 0
    for k in np.linspace(0, tot - 1, n).astype(int):
        cap.set(1, int(k)); ok, f = cap.read()
        if not ok: continue
        b, g, r = [f[..., i].astype(int) for i in range(3)]
        bl += int(((b > 150) & (r < 110) & (b - r > 80)).sum())
        rd += int(((r > 150) & (b < 110) & (r - b > 80) & (g < 110)).sum())
        wh += int(((b > 200) & (g > 200) & (r > 200)).sum())
    cap.release(); return tot, bl, rd, wh   # frames, blue, red, white

for clip in (442, 427, 467):
    for rep in ("skeleton", "overlay"):
        print(clip, rep, colour_pixels(D / "kaggle_representations" / str(clip) / f"{rep}.mp4"))

In [ ]:
import json
CHECK = Path("/kaggle/working/results/v2_check.jsonl")
run_gemma.run([442, 427, 467], ["raw_video", "skeleton"], ["R_rubric"], [1], out_path=CHECK)
for line in CHECK.read_text().splitlines():
    r = json.loads(line)
    print("=" * 70)
    print(r["instance"], r["rep"], "gt", r["true_class"], "pred", r["pred"], "ok", r["success"], r["error"])
    print(r["raw"])

In [ ]:
run_gemma.run(DEV_INSTANCES, ["raw_video", "skeleton", "overlay", "montage", "blind"],
              ["R_rubric", "Z_zero_shot"], [1])